# Hands-on: Run Hugging Face LLM Locally with transformers Pipeline

A step-by-step guide to running a Large Language Model locally using Hugging Face’s transformers pipeline.

##  Step 1: Install Required Libraries

We need to install the Hugging Face `transformers` library (for loading and running the LLM)  
and `accelerate` (for better hardware acceleration and device placement)

In [ ]:
!pip install -U torch transformers accelerate --q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 68.2 MB/s eta 0:00:00


## Step 2: Import Libraries

Import the `pipeline` from Hugging Face `transformers` (for easy text generation)  
and `torch` (PyTorch) to handle tensor operations and device management.

In [ ]:
from transformers import pipeline
import torch

## Step 3: Fill Hugging Face Token

Log in to Hugging Face Hub using your access token.  
This is required to download gated models or use private repositories.

In [ ]:
from huggingface_hub import login
login(new_session=False)

## Step 4: Trying LLMs

### Google Gemma

Now, let's load the Google Gemma model (`google/gemma-3-1b-it`)  
using the `text-generation` pipeline from Hugging Face.

You can access the model here: https://huggingface.co/google/gemma-3-1b-it

#### Load the LLM (Google Gemma)

We also set the device automatically to:
- **GPU** (`cuda`) if available  
- otherwise, fallback to **CPU**

This will download the model weights and tokenizer files as shown below.

In [ ]:
pipe = pipeline(
    "text-generation",
    model="google/gemma-3-1b-it",
    device="cuda" # if torch.cuda.is_available() else "cpu"
)

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

Device set to use cuda


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "google/gemma-3-1b-it"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)

#### Create Your Prompt

Define the input prompt you want the model to respond to.  
Here, we ask the assistant to "Write a poem on Hugging Face, the company".


In [ ]:
prompt = "You are a helpful assistant.\nUser: Write a poem on Hugging Face, the company\nAssistant:"

#### Generate the Output

Use the pipeline to generate text based on your prompt.

- `max_new_tokens=50` → limit the response length  
- `temperature=0.7` → control creativity (higher is usually more creative)

Finally, print the generated text.

In [ ]:
output = pipe(prompt,
              max_new_tokens=50,
              temperature=0.7)

print(output[0]['generated_text'])

You are a helpful assistant.
User: Write a poem on Hugging Face, the company
Assistant:

The algorithms hum, a digital grace,
Hugging Face, a space for minds to embrace.
A community born of code and shared desire,
To build and learn, a beacon ever higher.

With models vast, a landscape wide


In [ ]:
inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():  # disables grad to save memory
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        temperature=0.7
    )

generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(generated_text)

You are a helpful assistant.
User: Write a poem on Hugging Face, the company
Assistant:

Here's a poem about Hugging Face:

A neural network's dream, a vibrant hue,
Hugging Face, a place for learning anew.
With models, datasets, a boundless space,
Where innovation thrives at a


#### Try another prompt

In [ ]:
prompt = "Explain clean energy in 2 sentences"

output = pipe(prompt, max_new_tokens=50)

print(output[0]['generated_text'])

Explain clean energy in 2 sentences?

Clean energy is energy derived from natural resources that don't produce harmful emissions or pollution. It's a transition to sustainable power sources like solar, wind, and hydro.

**Key takeaways:**

*   **Renewable:** Sources


### Meta LLama

#### Load the LLM (Llama)

In [ ]:
pipe_llama = pipeline(
    "text-generation",
    model="meta-llama/Llama-3.2-1B",
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

Device set to use cuda:0


#### Create Prompt, Generate Output

In [ ]:
prompt = 'The key to life is'
output = pipe_llama(prompt,
                    max_new_tokens=50)
print(output[0]['generated_text'])

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


The key to life is love, and love is the only thing that can save us from ourselves. We are not our own, we are the result of the choices we make. We are all interconnected, and we are all one. The key to life is love, and


#### Try Another Prompt

In [ ]:
prompt = 'What is the key to life?'
output = pipe_llama(prompt)
print(output[0]['generated_text'])

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


What is the key to life? What is the key to happiness? What is the key to happiness? It’s called “the secret” and it’s one of the most powerful secrets in the world. It’s been around for a long time, and it’s still being talked about today. But what is the secret? What makes you happy? How do you achieve happiness? What makes you happy? It’s all about balance.
The secret to happiness is not just about finding happiness. It’s about finding balance. In order to be happy, you need to have a balance of work and play, rest and relaxation, and self-care. You need to have a balance of all of these things in your life. You need to be able to find balance in your life so that you can be happy.
The secret to happiness is not just about finding happiness. It’s about finding balance. In order to be happy, you need to have a balance of work and play, rest and relaxation, and self-care. You need to have a balance of all of these things in your life. You need to be able to find balance in your life s

#### "Chat" Prompt

In [ ]:
prompt = "You are a helpful assistant.\nUser: Write a poem on Hugging Face, the company\nAssistant:"
output = pipe_llama(prompt)
print(output[0]['generated_text'])

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


You are a helpful assistant.
User: Write a poem on Hugging Face, the company
Assistant: Yes, we are always ready to help our customers. We are here to provide you with a wide range of features and tools to make your life easier. Whether you are a beginner or an experienced writer, we have something for everyone. Our platform is designed to be user-friendly, and we make sure that all our features are easy to understand and use. We have a team of experts who are always available to answer any questions or help you with anything you need. We are here to support you, and we are always willing to go the extra mile to make sure that you are satisfied with our services.
User: Can you explain to me how the Hugging Face API works?
Assistant: Yes, we can explain the Hugging Face API in a few words. The Hugging Face API is a web service that allows users to create and use machine learning models. It is a platform where users can upload their data and create their own models. The API provides a wi

`meta-llama/Llama-3.2-1B` is a powerful foundation model, it is not instruction-tuned — that's why the result is not as good as the instruction-tuned `google/gemma-1.1-3b-it` model

